Because fraud is so rare, a naive model learns to predict "not fraud" 100% of the time and still achieves 99.8% accuracy, which is useless. I want apply SMOTE (Synthetic Minority Oversampling Technique) from imbalanced-learn to generate synthetic fraud samples in the training set only, never the test set.


## What SMOTE actually does


SMOTE, which stands for (Synthetic Minority Oversampling Technique), doesn't just copy fraud rows, it generates brand-new synthetic fraud samples by interpolating between existing ones. 

For each real fraud transaction, it finds its nearest fraud neighbors and creates new points along the line segments connecting them. 

This gives the model a richer, more varied view of what fraud looks like.

One critical rule: SMOTE only touches the training set. The test set stays completely untouched and imbalanced, because that reflects the real world your model will face in production.

In [ ]:
import sys
!"{sys.executable}" -m pip install imbalanced-learn

import os
os.makedirs('data/smote', exist_ok=True)

In [ ]:
from imblearn.over_sampling import SMOTE
import joblib

X_train = joblib.load('data/preprocessed/X_train.pkl')
y_train = joblib.load('data/preprocessed/y_train.pkl')

print("Before SMOTE:", y_train.value_counts().to_dict())
# {0: 227452, 1: 394}

smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

print("After SMOTE:", y_train_resampled.value_counts().to_dict())
# {0: 227452, 1: 227452}  ← now 50/50

joblib.dump(X_train_resampled, 'data/smote/X_train_resampled.pkl')
joblib.dump(y_train_resampled, 'data/smote/y_train_resampled.pkl')

Before SMOTE: {0: 227451, 1: 394}
After SMOTE: {0: 227451, 1: 227451}


['data/smote/y_train_resampled.pkl']

An alternative worth knowing: undersampling

Instead of creating more fraud rows, you can randomly remove legitimate rows until the classes are balanced. 

It's faster and uses less memory, but you throw away real data. For a dataset this large it's a reasonable trade-off.


In [ ]:
from imblearn.under_sampling import RandomUnderSampler

rus = RandomUnderSampler(random_state=42)
X_train_under, y_train_under = rus.fit_resample(X_train, y_train)
# Result: 394 fraud, 394 legit — tiny but balanced

print("After SMOTE:", y_train_under.value_counts().to_dict())
# {0: 227452, 1: 227452}  ← now 50/50

joblib.dump(X_train_under, 'data/smote/X_train_under.pkl')
joblib.dump(y_train_under, 'data/smote/y_train_under.pkl')

After SMOTE: {0: 394, 1: 394}


['data/smote/y_train_under.pkl']